# 1. Objective
The objective of this notebook is to clean, preprocess, and validate the bird monitoring dataset to ensure high data quality for subsequent Exploratory Data Analysis (EDA) and future Machine Learning (ML) models. This involves structured data consolidation, handling missing values with formal justifications, out-of-bounds validations, feature engineering, and constructing a comprehensive data quality report.


# 2. Data Sources
*   `Bird_Monitoring_Data_FOREST.XLSX`: Observations collected from forest habitats.
*   `Bird_Monitoring_Data_GRASSLAND.XLSX`: Observations collected from grassland habitats.
Data spans multiple years and involves different observer protocols.


In [3]:
import pandas as pd
import numpy as np
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

warnings.filterwarnings('ignore')


# 3. Data Loading and Consolidation
**Justification:** Instead of analyzing datasets individually, combining them provides a comprehensive view of species distributions across multiple ecosystems. We track the `Source_Sheet` to retain provenance.


In [4]:
forest_file = 'Bird_Monitoring_Data_FOREST.XLSX'
grassland_file = 'Bird_Monitoring_Data_GRASSLAND.XLSX'

def load_and_combine_excel(file_path):
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return pd.DataFrame()
    
    print(f"Reading data from: {file_path}")
    sheets_dict = pd.read_excel(file_path, sheet_name=None)
    
    df_list = []
    for sheet_name, df in sheets_dict.items():
        df['Source_Sheet'] = sheet_name
        df_list.append(df)
        
    combined_df = pd.concat(df_list, ignore_index=True)
    return combined_df

forest_df = load_and_combine_excel(forest_file)
grassland_df = load_and_combine_excel(grassland_file)

if not forest_df.empty and not grassland_df.empty:
    df_raw = pd.concat([forest_df, grassland_df], ignore_index=True)
    df = df_raw.copy()
    print(f"Total combined rows: {len(df)}")
else:
    print("Files not perfectly loaded. Check paths.")
    df = pd.DataFrame()


Reading data from: Bird_Monitoring_Data_FOREST.XLSX
Reading data from: Bird_Monitoring_Data_GRASSLAND.XLSX
Total combined rows: 17077


# Tracking Data Quality (Before/After)
We initialize metrics to generate a final Data Quality Report.


In [5]:
if not df.empty:
    initial_rows = len(df)
    print(initial_rows)
    initial_duplicates = df.duplicated().sum()
    print(initial_duplicates)
    initial_missing = df.isnull().sum()
    print(initial_missing)
    initial_missing = df.isnull().sum().sum()
    print(initial_missing)
    


17077
1705
Admin_Unit_Code                    0
Sub_Unit_Code                  16355
Site_Name                       8531
Plot_Name                          0
Location_Type                      0
Year                               0
Date                               0
Start_Time                         0
End_Time                           0
Observer                           0
Visit                              0
Interval_Length                    0
ID_Method                          2
Distance                        1486
Flyover_Observed                   0
Sex                             5183
Common_Name                        0
Scientific_Name                    0
AcceptedTSN                       33
NPSTaxonCode                    8531
AOU_Code                           0
PIF_Watchlist_Status               0
Regional_Stewardship_Status        0
Temperature                        0
Humidity                           0
Sky                                0
Wind                       

# 3.1 Cleaning Step: Handling Duplicates
**Justification:** Duplicate records can severely distort analysis and over-represent specific bird occurrences. They are completely removed rather than aggregated, as we assume them to be system entry artifacts.
**Assumption:** No two completely identical rows represent distinct valid observations made at the exact same time by the same observer.


In [6]:
if not df.empty:
    print(f"Duplicates before cleaning: {initial_duplicates}")
    df = df.drop_duplicates()
    print(f"Duplicates after cleaning: {df.duplicated().sum()}")


Duplicates before cleaning: 1705
Duplicates after cleaning: 0


# 3.2 Cleaning Step: Handling Missing Values
**Justification for Categorical:** We observed large numbers of missing values in categorical fields like `Sex` and `TaxonCode`. Instead of mode imputation (which would falsely assume the most frequent category), we use explicit assignment ("Unknown") to avoid introducing category bias into the dataset.



In [7]:
if not df.empty:
   # Categorical missing values
    fill_values = {
        'Sub_Unit_Code': 'Not Recorded',
        'Site_Name': 'Not Recorded',
        'Sex': 'Unknown',
        'ID_Method': 'Unknown',
        'AcceptedTSN': 'Unknown',
        'TaxonCode': 'Unknown',
        'NPSTaxonCode': 'Unknown',
        'Previously_Obs': 'Unknown',
        'Distance': 'Unknown'
    }
    df.fillna(value=fill_values, inplace=True)


# 4. Data Validation Checks
**Justification:** Ensures that domain constraints are met before analysis, acting as quality assurance testing.
*   **Ranges:** Temperature should be realistic within typical weather standards.
*   **Species Check:** Names shouldn't be empty or physically impossible variables.


In [8]:
if not df.empty:
    # Convert Data types appropriately
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df['Temperature'] = pd.to_numeric(df['Temperature'], errors='coerce')

    # Validation: Temperature range (-30 to 50 C degrees typical max boundaries)
    invalid_temps = df[(df['Temperature'] < -30) | (df['Temperature'] > 50)]
    print(f"Invalid Temperature records found that are out of typical bounds (-30, 50): {len(invalid_temps)}")
    df['Temperature'] = df['Temperature'].clip(lower=-30, upper=50) # Coerce outliers

    # Validation: Invalid common names
    invalid_names = df[df['Common_Name'].astype(str).str.strip() == '']
    print(f"Invalid/Empty common names found: {len(invalid_names)}")

    # Date ranges validation
    valid_dates = df['Date'].dt.year >= 2000
    print(f"Records with valid dates (>= 2000): {valid_dates.sum()} / {len(df)}")


Invalid Temperature records found that are out of typical bounds (-30, 50): 0
Invalid/Empty common names found: 0
Records with valid dates (>= 2000): 15372 / 15372


# 5. Feature Engineering
**Justification:** Extracting granular features from raw timestamps provides better categorical aggregations for our eventual models and insights.
*   `Month` and `Season`: Enable temporal pattern identification for migrations.
*   `Time_of_Day`: Groups activities into biological bins (Morning/Afternoon).
*   `Is_Forest`: Model-friendly numerical flag for habitat.


In [9]:
if not df.empty:
    # 1. Month Extraction
    df['Month'] = df['Date'].dt.month

    # 2. Season Evaluation
    def get_season(month):
        if pd.isnull(month): return 'Unknown'
        if month in [12, 1, 2]: return 'Winter'
        if month in [3, 4, 5]: return 'Spring'
        if month in [6, 7, 8]: return 'Summer'
        return 'Fall'
    df['Season'] = df['Month'].apply(get_season)

    # 3. Time-Based group
    df['Start_Time'] = df['Start_Time'].astype(str)
    def get_time_of_day(time_str):
        try:
            hour = int(str(time_str).split(':')[0])
            if 5 <= hour < 12: return 'Morning'
            elif 12 <= hour < 17: return 'Afternoon'
            else: return 'Evening/Night'
        except:
            return 'Unknown'
    df['Time_of_Day'] = df['Start_Time'].apply(get_time_of_day)

    # 4. Habitat type encoding
    df['Is_Forest'] = df['Location_Type'].apply(lambda x: 1 if str(x).lower().strip() == 'forest' else 0)

    # Peek at new features
    display(df[['Date', 'Month', 'Season', 'Start_Time', 'Time_of_Day', 'Location_Type', 'Is_Forest']].head())


,Date,Month,Season,Start_Time,Time_of_Day,Location_Type,Is_Forest
0,2018-05-22,5,Spring,06:19:00,Morning,Forest,1
1,2018-05-22,5,Spring,06:19:00,Morning,Forest,1
2,2018-05-22,5,Spring,06:19:00,Morning,Forest,1
3,2018-05-22,5,Spring,06:19:00,Morning,Forest,1
4,2018-05-22,5,Spring,06:19:00,Morning,Forest,1


# 6. Prepare Data for Future ML
**Justification:** Transforming variables shapes them for model ingestion. We need label encoders for specific targeted features and standard scalers for numeric inputs to ensure stable gradient descent logic.


In [2]:
# if not df.empty:
#     ml_df = df.copy()

#     # Label Encoding for Target Variable or crucial categorical variables
#     le = LabelEncoder()
#     ml_df['Location_Encoded'] = le.fit_transform(ml_df['Location_Type'].astype(str))
#     ml_df['Sky_Encoded'] = le.fit_transform(ml_df['Sky'].astype(str))

#     # Scaling Numeric
#     scaler = StandardScaler()
#     ml_df[['Temperature_Scaled', 'Distance_Scaled']] = scaler.fit_transform(ml_df[['Temperature', 'Distance']].fillna(0))

#     # Train-Test Split Readiness
#     features = ['Temperature_Scaled', 'Distance_Scaled', 'Is_Forest', 'Month', 'Sky_Encoded']
#     target = 'Location_Encoded'
#     X = ml_df[features].fillna(0)
#     y = ml_df[target]

#     X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
#     print("Data is now prepared and split for ML models!")
#     print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")


# 7. Data Quality Report
Below is the summary tracking of the major cleaning metrics to provide transparency on the pipeline modifications.


In [10]:
if not df.empty:
    final_rows = len(df)
    final_duplicates = df.duplicated().sum()
    final_missing = df.isnull().sum().sum()

    report = pd.DataFrame({
        'Metric': ['Total Rows', 'Missing Values', 'Duplicates'],
        'Before Cleaning': [initial_rows, initial_missing, initial_duplicates],
        'After Cleaning': [final_rows, final_missing, final_duplicates]
    })
    display(report)


,Metric,Before Cleaning,After Cleaning
0,Total Rows,17077,15372
1,Missing Values,57215,0
2,Duplicates,1705,0


# 8. Assumptions
1.  **Missing Data Randomness:** We assume that metadata missing from observations (e.g. `Sex`, `Distance`) was missing completely at random (MCAR) or missing at random (MAR), rather than being systematically excluded due to the observer or terrain.
2.  **Observation Errors:** We assume field data invariably contains some minor human observation errors, which were only corrected if they drastically fell outside logical programmatic bounds.
3.  **Environmental Variables:** Temperature bounds are assumed broadly consistent with the standard expected climate; slight outliers are clipped.

# 9. Limitations of Cleaning
1.  **Reduced Variability from Imputation:** Replacing numeric anomalies (like Distance) with Medians marginally compresses the data's natural variance, potentially minimizing the recorded effects of extreme spatial spotting.
2.  **Duplicate Dropping Nuance:** A portion of duplicate rows dropped might theoretically have been legitimate simultaneous identical recordings, losing minor frequency volume.
3.  **Categorical Fill Bias:** Using "Unknown" universally for empty textual observations retains structure but introduces an artificial category that ML systems must explicitly associate as 'no signal' rather than as a distinct class.


In [11]:
# Optional: Save back to DB or CSV based on original notebook pattern
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv
load_dotenv()
db_user, db_password, db_host, db_port, db_name = os.environ.get('DB_USER'), os.environ.get('DB_PASSWORD'), os.environ.get('DB_HOST'), os.environ.get('DB_PORT'), os.environ.get('DB_NAME')
connection_string = f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}'
try:
    engine = create_engine(connection_string)
    df.to_sql('bird_data', engine, if_exists='replace', index=False)
    print("SUCCESS: Cleaned dataset securely exported to PostgreSQL table.")
except Exception as e:
    print(f"DATABASE EXPORT ERROR: {e}")



DATABASE EXPORT ERROR: No module named 'psycopg2'


In [11]:
# Optional: Save back to SQLite database
import sqlite3
try:
    # Connect to (or create) an SQLite database file
    sqlite_conn = sqlite3.connect('bird_data.sqlite')
    
    # Save the dataframe to the 'bird_data' table
    df.to_sql('bird_data', sqlite_conn, if_exists='replace', index=False)
    print("SUCCESS: Cleaned dataset securely exported to SQLite database.")
    
    # Close the connection
    sqlite_conn.close()
except Exception as e:
    print(f"SQLITE EXPORT ERROR: {e}")

SUCCESS: Cleaned dataset securely exported to SQLite database.
